# PRICING_STRATEGY_INSIGHTS

- Done By: Kedar Raju Pawar
- date: 2026-02-26 time: 16:05:00

In [2]:
import pandas as pd

df= pd.read_csv(
    "../Value_Index_Design/value_index_model_ready.csv"
)
df_scaled=df.copy()
df_scaled.columns

Index(['Company', 'laptop_type', 'screen_size_inch',
       'display_resolution_width', 'display_resolution_height', 'panel_type',
       'is_touchscreen', 'resolution_class', 'cpu_brand', 'cpu_family',
       'cpu_generation', 'clock_speed_ghz', 'cpu_tier', 'gpu_brand', 'ram_gb',
       'ssd_gb', 'hdd_gb', 'flash_gb', 'hybrid_gb', 'total_storage_gb',
       'weight_kg', 'weight_class', 'price_band', 'Price', 'is_price_outlier',
       'Performance_Score', 'value_index', 'value_segment'],
      dtype='object')

## 1  — Creating Pricing Power Indicator

In [6]:
df_pricing = df_scaled.copy()

brand_summary = (
    df_pricing.groupby('Company')
    .agg({
        'value_index': 'mean',
        'Price': 'mean',
        'Performance_Score': 'mean'
    })
    .reset_index()
)

brand_summary.sort_values('Price', ascending=True)

,Company,value_index,Price,Performance_Score
17,VERO,0.000009,12331.656000,0.113333
11,MEDIACOM,0.000007,15717.600000,0.113214
3,CHUWI,0.000008,16745.726400,0.117778
0,ACER,0.000013,33461.812629,0.419413
5,FUJITSU,0.000014,38841.120000,0.547778
7,HP,0.000011,54775.399680,0.507184
9,LENOVO,0.000011,57480.870866,0.527809
2,ASUS,0.000010,58022.093371,0.526220
18,XIAOMI,0.000009,60390.882000,0.553333
16,TOSHIBA,0.000010,64967.812683,0.579228


## 2 — Identify Overpriced Brands

In [12]:
brand_summary['value_rank'] = brand_summary['value_index'].rank(ascending=False)
brand_summary['price_rank'] = brand_summary['Price'].rank(ascending=False)
brand_summary['price_rank'].value_counts()

price_rank
16.0    1
5.0     1
12.0    1
17.0    1
9.0     1
15.0    1
4.0     1
14.0    1
8.0     1
13.0    1
2.0     1
18.0    1
6.0     1
3.0     1
1.0     1
7.0     1
10.0    1
19.0    1
11.0    1
Name: count, dtype: int64

In [13]:
brand_summary['pricing_gap'] = (
    brand_summary['price_rank'] - brand_summary['value_rank']
)

In [14]:
brand_summary.sort_values('pricing_gap')

,Company,value_index,Price,Performance_Score,value_rank,price_rank,pricing_gap
14,RAZER,0.000005,201345.120000,0.791000,19.0,1.0,-18.0
10,LG,0.000006,111834.720000,0.674444,17.0,2.0,-15.0
6,GOOGLE,0.000006,89386.080000,0.520278,18.0,4.0,-14.0
1,APPLE,0.000007,88216.542212,0.557696,16.0,5.0,-11.0
12,MICROSOFT,0.000007,85903.788000,0.598333,15.0,6.0,-9.0
13,MSI,0.000008,95357.785976,0.683431,11.0,3.0,-8.0
15,SAMSUNG,0.000008,80332.920000,0.608646,12.0,7.0,-5.0
8,HUAWEI,0.000008,75870.720000,0.627500,10.0,8.0,-2.0
18,XIAOMI,0.000009,60390.882000,0.553333,8.0,11.0,3.0
16,TOSHIBA,0.000010,64967.812683,0.579228,7.0,10.0,3.0


## 3 — Identify Underpriced Brands

In [17]:
brand_summary.sort_values('pricing_gap',ascending=False)

,Company,value_index,Price,Performance_Score,value_rank,price_rank,pricing_gap
0,ACER,0.000013,33461.812629,0.419413,2.0,16.0,14.0
5,FUJITSU,0.000014,38841.120000,0.547778,1.0,15.0,14.0
7,HP,0.000011,54775.399680,0.507184,4.0,14.0,10.0
9,LENOVO,0.000011,57480.870866,0.527809,3.0,13.0,10.0
17,VERO,0.000009,12331.656000,0.113333,9.0,19.0,10.0
2,ASUS,0.000010,58022.093371,0.526220,6.0,12.0,6.0
4,DELL,0.000010,65963.045468,0.566561,5.0,9.0,4.0
11,MEDIACOM,0.000007,15717.600000,0.113214,14.0,18.0,4.0
3,CHUWI,0.000008,16745.726400,0.117778,13.0,17.0,4.0
18,XIAOMI,0.000009,60390.882000,0.553333,8.0,11.0,3.0


## 3 — Create Business Buckets

In [18]:
def pricing_strategy(row):
    if row['value_rank'] <= 5 and row['price_rank'] > 10:
        return "Underpriced - Can Increase Price"
    elif row['value_rank'] > 10 and row['price_rank'] <= 5:
        return "Premium - Brand Power Strategy"
    elif row['value_rank'] <= 5 and row['price_rank'] <= 5:
        return "Competitive Leader"
    else:
        return "Needs Repositioning"

brand_summary['pricing_strategy'] = brand_summary.apply(pricing_strategy, axis=1)

In [20]:
brand_summary.to_excel(
    "pricing_strategy_recommendations.xlsx",
    index=False
)